In [5]:
# 1. Import Libraries

In [3]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils import resample

In [2]:
%pip install nltk

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.5 MB ? eta -:--:--
   -------------------- ------------------- 0.8/1.5 MB 2.7 MB/s eta 0:00:01
   ---------------------------------- ----- 1.3/1.5 MB 2.4 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 2.3 MB/s  0:00:00

   ---------------------------------------- 0/4 [tqdm]
   ---------------------------------------- 0/4 [tqdm]
   ---------------------------------------- 0/4 [tqdm]
   ---------------------------------------- 0/4 [tqdm]
   ---------------------------------------- 0/4 [tqdm]
   ---------------------------------------- 0/4 [tqdm]
   ---------- ----------------------------- 1/4 [regex]
   ---------- ----------------------------- 1/4 [regex]
   -------------------- ------------------- 2/4 [click]
   -------------------- ------------------- 2/4 [click]
   -

In [6]:
# Download NLTK resources

In [4]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\chinm\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\chinm\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\chinm\AppData\Roaming\nltk_data...


True

In [7]:
# 2. Load Dataset

In [8]:
df = pd.read_csv("gender_dataset.csv")  # Make sure your CSV path is correct
print("Original dataset shape:", df.shape)
print(df['label'].value_counts())

Original dataset shape: (17880, 2)
label
biased      12691
unbiased     5189
Name: count, dtype: int64


In [9]:
# 3. Text Cleaning Function

In [10]:
def clean_text(text):
    text = text.lower()  # Lowercase
    text = re.sub(r'#URL_\S+', '', text)  # Remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove punctuation/numbers
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces
    return text

df['clean_text'] = df['text'].apply(clean_text)


In [11]:
# 4. Remove Stopwords

In [12]:
stop_words = set(stopwords.words('english'))
df['clean_text'] = df['clean_text'].apply(
    lambda x: ' '.join([w for w in x.split() if w not in stop_words])
)

In [13]:
# 5. Lemmatization

In [14]:
lemmatizer = WordNetLemmatizer()
df['clean_text'] = df['clean_text'].apply(
    lambda x: ' '.join([lemmatizer.lemmatize(w) for w in x.split()])
)


In [15]:
# 6. Handle Imbalanced Classes

In [16]:
df_majority = df[df.label=='biased']
df_minority = df[df.label=='unbiased']

# Upsample minority class
df_minority_upsampled = resample(
    df_minority,
    replace=True,
    n_samples=len(df_majority),
    random_state=42
)

df_balanced = pd.concat([df_majority, df_minority_upsampled])
print("Balanced dataset shape:", df_balanced.shape)
print(df_balanced['label'].value_counts())

Balanced dataset shape: (25382, 3)
label
biased      12691
unbiased    12691
Name: count, dtype: int64


In [17]:
# Choose columns to keep

In [18]:
df_balanced_to_save = df_balanced[['clean_text', 'label']]


In [19]:
# Save to CSV

In [20]:
df_balanced_to_save.to_csv("gender_dataset_preprocessed.csv", index=False)

print("Preprocessed dataset saved as 'gender_dataset_preprocessed.csv'")

Preprocessed dataset saved as 'gender_dataset_preprocessed.csv'
